In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch import nn
import warnings
from transformers import logging as hf_logging

hf_logging.set_verbosity_error()

np.random.seed(42)
torch.manual_seed(42)

In [8]:
print(torch.cuda.is_available())          # should print True
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [2]:
# Load data
train_df = pd.read_csv('full_train_df.csv')
test_df = pd.read_csv('full_test_df.csv')

# Create labels
train_df['label'] = (train_df['student_tag'] == '3 - Asking for More Information').astype(int)
test_df['label'] = (test_df['student_tag'] == '3 - Asking for More Information').astype(int)

# Calculate class weights
neg_count = (train_df['label'] == 0).sum()
pos_count = (train_df['label'] == 1).sum()
pos_weight = neg_count / pos_count
print(f"\nClass weight for positive class: {pos_weight:.2f}")
class_weights = torch.tensor([1.0, pos_weight], dtype=torch.float)


# Load model and tokenizer
model_name = "roberta-base"
tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Processs data to create input sequences for RoBERTa
def get_turn_num(row):
    if pd.notna(row['turn']):
        try:
            return int(float(row['turn']))
        except ValueError:
            return 'nan'
    return 'nan'


def create_input_text(row):
    turn_num = get_turn_num(row)

    prev = row['previous_context'] if pd.notna(row['previous_context']) else '(none)'
    subseq = row['subsequent_context'] if pd.notna(row['subsequent_context']) else '(none)'
    target = f"({turn_num}) [S] {row['student_utterance']}"

    # RoBERTa's tokenizer supports passing two sequences, which it separates with </s></s>
    # We use this to create a hard boundary between context and target+subsequent
    sequence_a = f"{prev}"
    sequence_b = f"[TARGET] {target} [AFTER] {subseq}"

    return sequence_a, sequence_b



Class weight for positive class: 28.97


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

In [3]:
# Prepare datasets
def make_dataset(df):
    seq_a, seq_b = zip(*df.apply(create_input_text, axis=1))
    return Dataset.from_dict({
        'seq_a': list(seq_a),
        'seq_b': list(seq_b),
        'label': df['label'].tolist()
    })

train_dataset = make_dataset(train_df)
test_dataset  = make_dataset(test_df)

lengths = pd.Series([
    len(tokenizer(a, b)['input_ids'])
    for a, b in zip(train_dataset['seq_a'], train_dataset['seq_b'])
])
print(lengths.describe())
print(f"Truncated (>512): {(lengths > 512).sum()} / {len(lengths)} ({100*(lengths > 512).mean():.1f}%)")

count    52394.000000
mean       251.261404
std         46.046789
min        151.000000
25%        219.000000
50%        244.000000
75%        275.000000
max        703.000000
dtype: float64
Truncated (>512): 47 / 52394 (0.1%)


In [4]:
def tokenize_function(examples):
    return tokenizer(
        examples['seq_a'],
        examples['seq_b'],
        padding='max_length',
        truncation=True,
        max_length=512
    )


train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset  = test_dataset.map(tokenize_function, batched=True)


train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])



Map:   0%|          | 0/52394 [00:00<?, ? examples/s]

Map:   0%|          | 0/7104 [00:00<?, ? examples/s]

In [5]:
# Custom Trainer with class-weighted loss
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = self.class_weights.to(logits.device)
        loss_fn = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [6]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions, zero_division=0),
        'recall': recall_score(labels, predictions, zero_division=0),
        'f1': f1_score(labels, predictions, zero_division=0)
    }

training_args = TrainingArguments(
    output_dir='./roberta_model',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_ratio=0.1,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',       # optimize for F1, not loss
    greater_is_better=True,
)

trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
print("\nStarting training...")
trainer.train()

print("\nEvaluating on test set...")
results = trainer.evaluate(test_dataset)
print("Test Results:")
for key, value in results.items():
    print(f"  {key}: {value:.4f}")

# Get and save predictions
predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)
test_df['predicted_label'] = pred_labels
test_df.to_csv('test_predictions.csv', index=False)

model.save_pretrained('./roberta_model_final')
tokenizer.save_pretrained('./roberta_model_final')
print("\nDone.")


Starting training...
{'loss': '0.6147', 'grad_norm': '5.552', 'learning_rate': '2.992e-07', 'epoch': '0.007634'}
{'loss': '0.6449', 'grad_norm': '6.335', 'learning_rate': '6.046e-07', 'epoch': '0.01527'}
{'loss': '0.562', 'grad_norm': '6.243', 'learning_rate': '9.099e-07', 'epoch': '0.0229'}
{'loss': '0.4739', 'grad_norm': '3.946', 'learning_rate': '1.215e-06', 'epoch': '0.03053'}
{'loss': '0.6138', 'grad_norm': '0.5634', 'learning_rate': '1.521e-06', 'epoch': '0.03817'}
{'loss': '1.266', 'grad_norm': '42.32', 'learning_rate': '1.826e-06', 'epoch': '0.0458'}
{'loss': '0.6268', 'grad_norm': '55.9', 'learning_rate': '2.131e-06', 'epoch': '0.05344'}
{'loss': '1.081', 'grad_norm': '0.1705', 'learning_rate': '2.437e-06', 'epoch': '0.06107'}
{'loss': '0.8142', 'grad_norm': '0.1771', 'learning_rate': '2.742e-06', 'epoch': '0.0687'}
{'loss': '0.9268', 'grad_norm': '0.1849', 'learning_rate': '3.047e-06', 'epoch': '0.07634'}
{'loss': '1.201', 'grad_norm': '0.3201', 'learning_rate': '3.353e-06',